les imports

In [30]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage

In [5]:
load_dotenv(override=True)

True

In [49]:
loader= PyPDFLoader("CV 4.pdf")

In [9]:
tokenizer =tiktoken.encoding_for_model("gpt-4o-mini")

In [10]:
splitter= RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=50
)

In [13]:
chunks= loader.load_and_split(splitter)

In [14]:
print(len(chunks))

3


In [15]:
print(chunks[0])

page_content='Omar Tarrouzi
E.M.S.I — Ingénierie IA & Science des Données
omartarrouzicontact@gmail.com | 0639 629 030 | Casablanca, Maroc
linkedin.com/in/omar-tarrouzi | github.com/Omar-Tarrouzi
Profil
Étudiant en 4ème année du cycle d’ingénierie en Intelligence Artificielle et Science des Données. Passionné par
la conception de solutions basées sur les données, je m’intéresse au développement des applications intégrant le
machine learning, le traitement de données et des systèmes scalables. À la recherche d’un stage en Data Science,
IA ou Data Engineering.
Formation
Cycle Ingénierie — Intelligence Artificielle & Science des Données2022 – en cours
École Marocaine des Sciences de l’Ingénieur (E.M.S.I), Casablanca4 ème année
Expérience professionnelle
Développeur Web — StageArtiflow, Casablanca
Refonte du site vitrine & optimisation des performancesJuillet-Août 2025
— Conception et développement du site sous WordPress/Elementor, hébergé sur Hostinger.
— Déploiement d’Optiplus (projet pe

tout stocker dans la bd

In [19]:
embedding_model=OpenAIEmbeddings()

In [20]:
vector_store=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="cv_data_collection"
    persist_directory="cv_data_collection"
)

In [21]:
retriever=vector_store.as_retriever(search_kwargs={"k":10})

In [45]:
def retriever_tool(query:str)->str:
    """
     permet de chercher des infos sur des candidats
     comme nom, prenom et diplome dans cv 4
    """
    relevent_documents=retriever.invoke(query)
    context_list=[d.page_content for d in relevent_documents]
    context= ".".join(context_list)
    return context

In [62]:
llm=ChatOpenAI(model="gpt-4o-mini",temperature=0)
agent=  create_agent(
model =llm,
tools=[retriever_tool,get_company_info],
system_prompt="Repond à la question de l'utilisateur avec les tools fournit "

)

In [63]:
resp =agent.invoke(input={
    "messages":[
        HumanMessage("Nom,prenom,diplome d'omar  et les infos sur l'entreprise")
    ]
})

In [64]:
print(resp['messages'][-1].content)

Voici les informations concernant Omar :

- **Nom** : Tarrouzi
- **Prénom** : Omar
- **Diplôme** : E.M.S.I — Ingénierie IA & Science des Données

### Profil
Omar est étudiant en 4ème année du cycle d’ingénierie en Intelligence Artificielle et Science des Données. Il est passionné par la conception de solutions basées sur les données et s'intéresse au développement d'applications intégrant le machine learning, le traitement de données et des systèmes scalables. Il est à la recherche d’un stage en Data Science, IA ou Data Engineering.

### Informations sur l'entreprise
- **Nom de l'entreprise** : l'entreprise
- **Domaine** : IT
- **Chiffre d'affaires** : 120,870,000

Si vous avez besoin de plus d'informations, n'hésitez pas à demander !


In [60]:
def get_company_info (companyname:str):
    """"
    consulter des infos sur l'entreprise donnée
    """
    return{
        "companyname":companyname,
        "domain":"IT",
        "turnover":120_870_000
    }

In [67]:
from IPython.display import Markdown, display

print(display(Markdown(resp['messages'][-1].content)))

Voici les informations concernant Omar :

- **Nom** : Tarrouzi
- **Prénom** : Omar
- **Diplôme** : E.M.S.I — Ingénierie IA & Science des Données

### Profil
Omar est étudiant en 4ème année du cycle d’ingénierie en Intelligence Artificielle et Science des Données. Il est passionné par la conception de solutions basées sur les données et s'intéresse au développement d'applications intégrant le machine learning, le traitement de données et des systèmes scalables. Il est à la recherche d’un stage en Data Science, IA ou Data Engineering.

### Informations sur l'entreprise
- **Nom de l'entreprise** : l'entreprise
- **Domaine** : IT
- **Chiffre d'affaires** : 120,870,000

Si vous avez besoin de plus d'informations, n'hésitez pas à demander !

None
